<a href="https://colab.research.google.com/github/olorunfemibabalola/BMI-Prediction-CV/blob/main/08_Better_CV_Kaggle_submission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# BMI Prediction

## Task 1: Download the dataset

You can access the competition data and submit your solutions via the [Kaggle website](https://www.kaggle.com/c/budl25) or using the [Kaggle API](https://github.com/Kaggle/kaggle-api). In the latter case all the interactions with Kaggle can be performed without leaving the notebook environment, so this is what we're going to use.

In [23]:
# put Kaggle API credentials where they belong
!mkdir -p /root/.config/kaggle
!echo '{"username":"olorunfemibabalola","key":"7c91bedda7863e6f34397263ce7dd3d4"}' > /root/.config/kaggle/kaggle.json
#!echo '{"username":"YOUR_USERNAME_GOES_HERE","key":"YOUR_KEY_GOES_HERE"}' > /root/.config/kaggle/kaggle.json
!chmod 600 /root/.config/kaggle/kaggle.json



In the cell below you will download the competition data and unzip it:

In [24]:
import kaggle
!kaggle competitions download -c bucv26 --force
!unzip -o *.zip

100% 36.8M/36.8M [00:00<00:00, 74.2MB/s]

Archive:  bucv26.zip
  inflating: sample_submission.csv   
  inflating: test_images/1144_7090.jpg  
  inflating: test_images/1148_7947.jpg  
  inflating: test_images/1290_2619.jpg  
  inflating: test_images/1301_4384.jpg  
  inflating: test_images/1334_1570.jpg  
  inflating: test_images/1455_3146.jpg  
  inflating: test_images/1513_1725.jpg  
  inflating: test_images/1590_4958.jpg  
  inflating: test_images/1597_3761.jpg  
  inflating: test_images/1651_2690.jpg  
  inflating: test_images/1733_5951.jpg  
  inflating: test_images/1750_4226.jpg  
  inflating: test_images/1758_4712.jpg  
  inflating: test_images/1998_7549.jpg  
  inflating: test_images/1999_9904.jpg  
  inflating: test_images/2049_8804.jpg  
  inflating: test_images/2084_9094.jpg  
  inflating: test_images/2092_7682.jpg  
  inflating: test_images/2143_5607.jpg  
  inflating: test_images/2169_7528.jpg  
  inflating: test_images/2171_1150.jpg  
  inflating: test_images/2193_5298.jpg

In [25]:
!ls /content/

bucv26.zip   sample_submission.csv  test_images   train_labels.csv
sample_data  submission.csv	    train_images


Let's load the .csv file and see what it looks like:

In [26]:
import pandas as pd
df = pd.read_csv('train_labels.csv')
df.head(5)

,anon_id,sex,bmi,filename
0,5956,0,21.7,5956_8270.jpg
1,5956,0,21.7,5956_1860.jpg
2,5956,0,21.7,5956_6390.jpg
3,5956,0,21.7,5956_6191.jpg
4,5956,0,21.7,5956_6734.jpg


In [27]:
train_dir = '/content/train_images'
test_dir = '/content/test_images'

## Task 2: Prepare data for training

We afre not downsizing the images anymore; higher resolution often allows to build a better model.

Here we create a fairly standard and minimalistic custom Pytorch dataset.

In [28]:
import os
from tqdm.auto import tqdm
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.transforms.v2 import Normalize, Compose, ToImage, ToDtype, Resize

class XrayDataset(Dataset):
    def __init__(self, img_dir, df_labels=None,transform=None):
        self.img_dir = img_dir
        self.df_labels = df_labels
        self.transform = transform

        self.img_files = os.listdir(img_dir)
        if df_labels is not None:
            self.file2label = dict(zip(df_labels['filename'], zip(df_labels['sex'], df_labels['bmi'])))

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        image = Image.open(f'{self.img_dir}/{img_name}').convert('L') # our images are greyscale
        if self.transform: image = self.transform(image)
        label = self.file2label[img_name] if self.df_labels is not None else [-1, -1]
        return image, torch.tensor(label[0], dtype=torch.float32), torch.tensor(label[1], dtype=torch.float32)

In [29]:
# You'll need to calculate the mean and std deviation of your dataset for proper normalization!
transforms = Compose([
    ToImage(),                           # Convert a tensor, ndarray, or PIL Image to torchvision.tv_tensors.Image
    Resize(224),                         # Images should all be 224x224 already but since it's not guranteed, we resize here
    ToDtype(torch.float32, scale=True),  # Converts the input to a specific dtype, optionally scaling the values to 0..1 range
    Normalize(mean=[0.5], std=[0.5])     # Normalization, currently using placeholder values!
])

# transforms is where you could add data augmentations too (like random flip or rotation) but for training data only!
ds = XrayDataset('/content/train_images', df, transform=transforms)
ds[0]

(Image([[[-1.0000, -1.0000, -1.0000,  ..., -0.7725, -0.7725, -0.7725],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9216, -0.9216, -0.9216],
         [-1.0000, -1.0000, -1.0000,  ..., -0.9922, -0.9922, -0.9922],
         ...,
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000],
         [-1.0000, -1.0000, -1.0000,  ..., -1.0000, -1.0000, -1.0000]]], ),
 tensor(0.),
 tensor(23.1000))

### Calculate Mean and Standard Deviation for Normalization

Proper normalization can significantly improve model performance. Let's calculate the mean and standard deviation of your training images to use in the `Normalize` transform.

In [30]:
from torch.utils.data import DataLoader

# Create a temporary DataLoader for calculating mean and std
# We'll use a DataLoader without normalization first to get raw pixel values
# Also, batch size of 1 is used to ensure all images are processed sequentially without partial batches affecting mean calculation
calc_ds = XrayDataset('/content/train_images', df, transform=Compose([
    ToImage(),
    Resize(224),
    ToDtype(torch.float32, scale=True)
]))
calc_loader = DataLoader(calc_ds, batch_size=1, shuffle=False)

mean = 0.
std = 0.
nb_samples = 0
for images, _, _ in tqdm(calc_loader, desc="Calculating Mean and Std"): # only need images
    batch_samples = images.size(0)
    images = images.view(batch_samples, images.size(1), -1)
    mean += images.mean(2).sum(0)
    std += images.std(2).sum(0)
    nb_samples += batch_samples

mean /= nb_samples
std /= nb_samples

print(f"Calculated Mean: {mean.item():.4f}, Calculated Std: {std.item():.4f}")

# Update the transforms with the calculated mean and std
transforms = Compose([
    ToImage(),
    Resize(224),
    ToDtype(torch.float32, scale=True),
    Normalize(mean=[mean.item()], std=[std.item()])
])

# Re-create datasets and dataloaders with the updated transforms
ds = XrayDataset('/content/train_images', df, transform=transforms)
test_ds  = XrayDataset('/content/test_images', transform=transforms)

# Re-create subsets (train_ds, valid_ds) using the same indices to maintain split
train_ds = Subset(ds, train_ixs)
valid_ds = Subset(ds, valid_ixs)

# Re-create dataloaders
train_loader = DataLoader(train_ds, batch_size=bs,   shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=2*bs, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=2*bs, shuffle=False)

print("Datasets and DataLoaders re-initialized with proper normalization values.")

Calculating Mean and Std:   0%|          | 0/1000 [00:00<?, ?it/s]

Calculated Mean: 0.4093, Calculated Std: 0.3558
Datasets and DataLoaders re-initialized with proper normalization values.


In [31]:
import random

train_size = len(ds)
valid_size = int(0.2 * train_size)

all_indices = list(range(train_size))
valid_ixs = random.sample(all_indices, valid_size)
train_ixs = list(set(all_indices) - set(valid_ixs))

train_ds = Subset(ds, train_ixs)
valid_ds = Subset(ds, valid_ixs)
test_ds  = XrayDataset('/content/test_images', transform=transforms)  # we don't have the labels, the goal is to predict them

train_ds[0] # notice we're now getting a normalized tensor

(Image([[[-1.1502, -1.1502, -1.1502,  ..., -0.8306, -0.8306, -0.8306],
         [-1.1502, -1.1502, -1.1502,  ..., -1.0400, -1.0400, -1.0400],
         [-1.1502, -1.1502, -1.1502,  ..., -1.1392, -1.1392, -1.1392],
         ...,
         [-1.1502, -1.1502, -1.1502,  ..., -1.1502, -1.1502, -1.1502],
         [-1.1502, -1.1502, -1.1502,  ..., -1.1502, -1.1502, -1.1502],
         [-1.1502, -1.1502, -1.1502,  ..., -1.1502, -1.1502, -1.1502]]], ),
 tensor(0.),
 tensor(23.1000))

In [32]:
# Create dataloaders
bs = 48
train_loader = DataLoader(train_ds, batch_size=bs,   shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=2*bs, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=2*bs, shuffle=False)

bx, by_c, by_r = next(iter(train_loader))
bx.shape, by_c.shape, by_r.shape

(torch.Size([48, 1, 224, 224]), torch.Size([48]), torch.Size([48]))

## Task 3: Model and training

In [33]:
# train on GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

Here we are now using a simple CNN from the lectures:

In [34]:
import torch.nn as nn

# simple Convolutional Neural Net (CNN)
class SimpleCNN(nn.Module):
    def __init__(self, hidden_size=64):
        super().__init__()

        # simple CNN from the lecture notebook
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels=1,  out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.MaxPool2d(kernel_size=2, stride=2),                                                          # [bs,16,h/2,w/2]
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.AdaptiveAvgPool2d((4,4)),                                                                    # [bs,32,4,4]
            nn.Flatten()                                                                                    # [bs,32*4*4]
        )

        # this head will be used for the classification task (sex 0 or 1)
        self.clasf_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),  # we only have two classes, so we could have either 2 or 1 output
            nn.Sigmoid()
        )

        # this head will be used for the BMI regression task
        # note that although it has almost the same architecture as the classification head,
        # it is a separate module with separate parameters!
        self.regr_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, 1),  # we only have one regression output (predicted BMI)
        )

    def forward(self, x):
        out = self.backbone(x)
        out_clasf = self.clasf_head(out)
        out_regr = self.regr_head(out)
        return out_clasf, out_regr  # not we have *two* outputs here!

# Instantiate the model
model = SimpleCNN(hidden_size=64)
model.to(device)
model

SimpleCNN(
  (backbone): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): LeakyReLU(negative_slope=0.01)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): LeakyReLU(negative_slope=0.01)
    (7): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): LeakyReLU(negative_slope=0.01)
    (9): AdaptiveAvgPool2d(output_size=(4, 4))
    (10): Flatten(start_dim=1, end_dim=-1)
  )
  (clasf_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
    (3): Sigmoid()
  )
  (regr_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out

### Add Dropout to the Model for Regularization

To prevent overfitting and help the model generalize better, we can add `Dropout` layers to the neural network. Dropout randomly sets a fraction of input units to zero at each update during training, which helps to break co-adaptations between neurons.

In [35]:
import torch.nn as nn

# simple Convolutional Neural Net (CNN)
class SimpleCNN(nn.Module):
    def __init__(self, hidden_size=64, dropout_rate=0.5):
        super().__init__()

        # simple CNN from the lecture notebook
        self.backbone = nn.Sequential(
            nn.Conv2d(in_channels=1,  out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.Conv2d(in_channels=16, out_channels=16, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,16,h,w]
            nn.MaxPool2d(kernel_size=2, stride=2),                                                          # [bs,16,h/2,w/2]
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, stride=1, padding=1), nn.LeakyReLU(), # [bs,32,h/2,w/2]
            nn.AdaptiveAvgPool2d((4,4)),                                                                    # [bs,32,4,4]
            nn.Flatten()                                                                                    # [bs,32*4*4]
        )

        # this head will be used for the classification task (sex 0 or 1)
        self.clasf_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # Added Dropout
            nn.Linear(hidden_size, 1),  # we only have two classes, so we could have either 2 or 1 output
            nn.Sigmoid()
        )

        # this head will be used for the BMI regression task
        # note that although it has almost the same architecture as the classification head,
        # it is a separate module with separate parameters!
        self.regr_head = nn.Sequential(
            nn.Linear(32*4*4, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate), # Added Dropout
            nn.Linear(hidden_size, 1),  # we only have one regression output (predicted BMI)
        )

    def forward(self, x):
        out = self.backbone(x)
        out_clasf = self.clasf_head(out)
        out_regr = self.regr_head(out)
        return out_clasf, out_regr  # not we have *two* outputs here!

# Re-instantiate the model with dropout
model = SimpleCNN(hidden_size=64, dropout_rate=0.5)
model.to(device)
print(model)
print("Model re-instantiated with Dropout layers.")

SimpleCNN(
  (backbone): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): LeakyReLU(negative_slope=0.01)
    (2): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): LeakyReLU(negative_slope=0.01)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): LeakyReLU(negative_slope=0.01)
    (7): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): LeakyReLU(negative_slope=0.01)
    (9): AdaptiveAvgPool2d(output_size=(4, 4))
    (10): Flatten(start_dim=1, end_dim=-1)
  )
  (clasf_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(in_features=64, out_features=1, bias=True)
    (4): Sigmoid()
  )
  (regr_head): Sequential(
    (0): Linear(in_features=512, out_features=64, bias=True)
    (1): ReL

### Exploring a More Complex Model Architecture (ResNet-like)

To address the high validation loss, let's explore a more complex model architecture. We will modify the `SimpleCNN` to incorporate more convolutional layers and introduce residual connections, inspired by ResNet architectures. Residual connections help in training deeper networks by allowing gradients to flow more easily through the network.

Here's how we'll modify the `SimpleCNN`:

1.  **Increased Depth:** More convolutional layers will be added to extract richer features.
2.  **Residual Blocks:** We'll define a `ResidualBlock` class, which includes two convolutional layers and a shortcut connection. This allows the network to learn residual functions, potentially improving optimization and performance.
3.  **Integration:** These residual blocks will replace some of the simple convolutional layers in the `SimpleCNN`'s backbone.

In [46]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, downsample=None):
        super(ResidualBlock, self).__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.LeakyReLU()
        )
        self.conv2 = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(out_channels)
        )
        self.downsample = downsample
        self.leaky_relu = nn.LeakyReLU()
        self.out_channels = out_channels

    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.leaky_relu(out)
        return out

class ResNetCNN(nn.Module):
    def __init__(self, hidden_size=64, dropout_rate=0.5):
        super().__init__()

        self.in_channels = 16
        self.conv_initial = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.LeakyReLU()
        )
        self.layer1 = self.make_layer(ResidualBlock, 16, 2, stride=1)
        self.layer2 = self.make_layer(ResidualBlock, 32, 2, stride=2)
        self.layer3 = self.make_layer(ResidualBlock, 64, 2, stride=2)

        self.backbone = nn.Sequential(
            self.conv_initial,
            self.layer1,
            self.layer2,
            self.layer3,
            nn.AdaptiveAvgPool2d((4,4)),
            nn.Flatten()
        )

        self.clasf_head = nn.Sequential(
            nn.Linear(64*4*4, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1),
            nn.Sigmoid()
        )

        self.regr_head = nn.Sequential(
            nn.Linear(64*4*4, hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(hidden_size, 1),
        )

    def make_layer(self, block, out_channels, blocks, stride=1):
        downsample = None
        if stride != 1 or self.in_channels != out_channels:
            downsample = nn.Sequential(
                nn.Conv2d(self.in_channels, out_channels, kernel_size=3, stride=stride, padding=1),
                nn.BatchNorm2d(out_channels),
            )
        layers = []
        layers.append(block(self.in_channels, out_channels, stride, downsample))
        self.in_channels = out_channels
        for i in range(1, blocks):
            layers.append(block(out_channels, out_channels))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = self.backbone(x)
        out_clasf = self.clasf_head(out)
        out_regr = self.regr_head(out)
        return out_clasf, out_regr

# Re-instantiate the model with the new ResNetCNN architecture
model = ResNetCNN(hidden_size=64, dropout_rate=0.5)
model.to(device)
print(model)
print("Model re-instantiated with ResNet-like architecture.")

ResNetCNN(
  (conv_initial): Sequential(
    (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.01)
  )
  (layer1): Sequential(
    (0): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): LeakyReLU(negative_slope=0.01)
      )
      (conv2): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (leaky_relu): LeakyReLU(negative_slope=0.01)
    )
    (1): ResidualBlock(
      (conv1): Sequential(
        (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affin

Now that we have a more complex model, let's redefine the optimizer and scheduler to ensure they are initialized with the new model's parameters. We also re-initialize the `min_valid_loss` and `epochs_no_improve` for early stopping.

In [47]:
# Re-initialize the model and optimizer
model = ResNetCNN(hidden_size=64, dropout_rate=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Re-initialize the learning rate scheduler
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

# Re-initialize early stopping variables
min_valid_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

print("Model, optimizer, scheduler, and early stopping variables re-initialized.")

Model, optimizer, scheduler, and early stopping variables re-initialized.


In [36]:
import torch.nn as nn
import torch.optim as optim

# loss and optimiser
criterion_clasf = nn.BCELoss()
criterion_regr = nn.L1Loss()

model = SimpleCNN(hidden_size=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

### Add Learning Rate Scheduler

A learning rate scheduler adjusts the learning rate during training. This can help the model converge faster and achieve a lower validation loss by allowing larger steps early in training and smaller, more precise steps later on.

In [37]:
import torch.optim.lr_scheduler as lr_scheduler

# Initialize the model and optimizer again, as the model was redefined
model = SimpleCNN(hidden_size=64, dropout_rate=0.5).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Add a learning rate scheduler (e.g., StepLR, ReduceLROnPlateau)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=10)

print("Learning rate scheduler initialized.")

Learning rate scheduler initialized.


In [38]:
def one_epoch(model, loader, criterion_clasf, criterion_regr, optimizer=None):
    device = next(model.parameters()).device

    if optimizer is None:
        model.eval()
    else:
        model.train()

    running_loss = 0.0
    total_predictions = 0
    correct_predictions = 0
    losses = []

    with torch.set_grad_enabled(optimizer is not None): # like torch.no_grad() but only if no optimizer given

        for images, labels_c, labels_r in loader:
            images, labels_c, labels_r = images.to(device), labels_c.to(device), labels_r.to(device)

            # Forward pass
            outputs = model(images)
            loss_clasf = criterion_clasf(outputs[0], labels_c.unsqueeze(1))
            loss_regr = criterion_regr(outputs[1], labels_r.unsqueeze(1))
            loss = loss_clasf*10 + loss_regr
            # print(f"classification loss: {loss_clasf.item():0.4f}, regression loss: {loss_regr.item():0.2f}, ratio: {(loss_regr/loss_clasf).item():0.2f}")

            # Backward and optimize
            if optimizer is not None:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_predictions += images.size(0)
            running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / total_predictions

    return epoch_loss

In [39]:
# training loop
num_epochs = 100

train_losses, valid_losses = [], []

for epoch in range(num_epochs):

    train_loss = one_epoch(model, train_loader, criterion_clasf, criterion_regr, optimizer)
    valid_loss = one_epoch(model, valid_loader, criterion_clasf, criterion_regr)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}], Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}')

Epoch [01/100], Training Loss: 25.4834, Validation Loss: 14.9596
Epoch [02/100], Training Loss: 12.9304, Validation Loss: 9.6159
Epoch [03/100], Training Loss: 11.8889, Validation Loss: 9.7122
Epoch [04/100], Training Loss: 11.5328, Validation Loss: 9.9173
Epoch [05/100], Training Loss: 11.6488, Validation Loss: 9.5119
Epoch [06/100], Training Loss: 11.3743, Validation Loss: 9.8851
Epoch [07/100], Training Loss: 11.5026, Validation Loss: 10.0449
Epoch [08/100], Training Loss: 11.4489, Validation Loss: 9.3188
Epoch [09/100], Training Loss: 11.1992, Validation Loss: 9.7851
Epoch [10/100], Training Loss: 11.2694, Validation Loss: 9.2341
Epoch [11/100], Training Loss: 11.0299, Validation Loss: 9.3217
Epoch [12/100], Training Loss: 11.1056, Validation Loss: 9.1705
Epoch [13/100], Training Loss: 11.0701, Validation Loss: 10.0645
Epoch [14/100], Training Loss: 10.8498, Validation Loss: 8.7453
Epoch [15/100], Training Loss: 10.7069, Validation Loss: 8.4413
Epoch [16/100], Training Loss: 10.018

### Training Loop with Learning Rate Scheduler

Now, let's modify the training loop to incorporate the learning rate scheduler.

In [48]:
# training loop with scheduler and early stopping
num_epochs = 150
patience = 20 # Number of epochs to wait before early stopping if no improvement
min_valid_loss = float('inf')
epochs_no_improve = 0
best_model_state = None

train_losses, valid_losses = [], []

for epoch in range(num_epochs):

    train_loss = one_epoch(model, train_loader, criterion_clasf, criterion_regr, optimizer)
    valid_loss = one_epoch(model, valid_loader, criterion_clasf, criterion_regr)

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    # Step the scheduler based on validation loss
    scheduler.step(valid_loss)

    print(f'Epoch [{epoch+1:02d}/{num_epochs}], Training Loss: {train_loss:.4f}, Validation Loss: {valid_loss:.4f}, Current LR: {optimizer.param_groups[0]["lr"]:.6f}')

    # Early stopping logic
    if valid_loss < min_valid_loss:
        min_valid_loss = valid_loss
        epochs_no_improve = 0
        best_model_state = model.state_dict() # Save the best model state
        print(f'\tValidation loss improved! Saving model with loss: {min_valid_loss:.4f}')
    else:
        epochs_no_improve += 1
        print(f'\tValidation loss did not improve for {epochs_no_improve} epoch(s).')
        if epochs_no_improve == patience:
            print(f'Early stopping triggered after {epoch+1} epochs.')
            break

# Load the best model weights after training (or early stopping)
if best_model_state:
    model.load_state_dict(best_model_state)
    print('Loaded best model weights based on validation loss.')


Epoch [01/150], Training Loss: 16.7305, Validation Loss: 10.6719, Current LR: 0.001000
	Validation loss improved! Saving model with loss: 10.6719
Epoch [02/150], Training Loss: 9.1398, Validation Loss: 23.1513, Current LR: 0.001000
	Validation loss did not improve for 1 epoch(s).
Epoch [03/150], Training Loss: 8.2951, Validation Loss: 6.7094, Current LR: 0.001000
	Validation loss improved! Saving model with loss: 6.7094
Epoch [04/150], Training Loss: 7.3630, Validation Loss: 10.4099, Current LR: 0.001000
	Validation loss did not improve for 1 epoch(s).
Epoch [05/150], Training Loss: 6.9867, Validation Loss: 9.0728, Current LR: 0.001000
	Validation loss did not improve for 2 epoch(s).
Epoch [06/150], Training Loss: 6.7642, Validation Loss: 13.2866, Current LR: 0.001000
	Validation loss did not improve for 3 epoch(s).
Epoch [07/150], Training Loss: 6.7137, Validation Loss: 17.5808, Current LR: 0.001000
	Validation loss did not improve for 4 epoch(s).
Epoch [08/150], Training Loss: 6.5527

### Other Potential Methods to Reduce Validation Loss:

1.  **Early Stopping:** Stop training when the validation loss stops improving for a certain number of epochs (patience). This prevents overfitting to the training data.
2.  **More Complex Model Architecture:** The current CNN is quite simple. You could explore:
    *   Adding more convolutional layers.
    *   Using different kernel sizes or strides.
    *   Incorporating residual connections (ResNet-like blocks).
    *   Experimenting with pre-trained models (e.g., ResNet, VGG, EfficientNet) and fine-tuning them on your dataset. This often leads to significant improvements, especially with smaller datasets.
3.  **Data Augmentation:** While you have `Resize` and `ToDtype` in your transforms, more advanced data augmentations can help the model see more diverse training examples and improve generalization. Consider `RandomHorizontalFlip`, `RandomRotation`, `ColorJitter`, etc. (but only for the training dataset).
4.  **Hyperparameter Tuning:** Systematically search for optimal values for:
    *   Learning rate (`lr`).
    *   Batch size (`bs`).
    *   `hidden_size` in your `SimpleCNN`.
    *   `dropout_rate`.
    *   The loss weighting (the `10` in `loss = loss_clasf*10 + loss_regr`).
5.  **Different Optimizers:** Experiment with other optimizers like SGD with momentum, RMSprop, etc.
6.  **Loss Function Adjustment:** If one task is more important or harder, you might adjust the weighting between classification and regression losses (`loss_clasf*10 + loss_regr`).
7.  **Larger Dataset:** If feasible, more training data generally leads to better generalization and lower validation loss.

## Task 4: Predictions for the holdout (i.e. the one held on Kaggle) dataset

In [50]:
import numpy as np

model.eval()  # Set the model to evaluation mode
hold_out_pred_sex, hold_out_pred_bmi = [], []

with torch.no_grad():
    for images, _, _ in test_loader: # test_loader does not have labels
        images = images.to(device)
        outputs = model(images)
        predicted_sex = (outputs[0] > 0.5).float().cpu().numpy() # Apply threshold and convert to numpy
        predicted_bmi = outputs[1].cpu().numpy()

        hold_out_pred_sex.extend(predicted_sex.flatten().tolist())
        hold_out_pred_bmi.extend(predicted_bmi.flatten().tolist())

# Get the filenames for the test set
hold_out_filenames = test_ds.img_files

# Create a DataFrame for submission
submission_df = pd.DataFrame({'filename': hold_out_filenames, 'sex': hold_out_pred_sex, 'bmi': hold_out_pred_bmi})

# Kaggle submission requires the header and index=False
submission_df.to_csv('submission.csv', index=False)

print("Submission file created successfully!")
print(submission_df.head())

Submission file created successfully!
        filename  sex        bmi
0  9177_2880.jpg  1.0  22.139753
1  5185_4889.jpg  0.0  18.796293
2  4202_7304.jpg  0.0  25.358770
3  1597_3761.jpg  1.0  23.013441
4  1455_3146.jpg  0.0  18.475904


## Task 5: Submitting to Kaggle

You can submit your predications via the Kaggle website or using the API. Either way, don’t forget to **add a brief description of the submission** before you upload. See how the submission stands on the leaderboard.

In [52]:
!kaggle competitions submit -c bucv26 -f submission.csv -m 'ResNetCNN with normalization, dropout, and learning rate scheduling'



100% 6.98k/6.98k [00:00<00:00, 34.2kB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission
